In [10]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Hp\Desktop\sentiment\Twitter-and-Reddit-Sentimental-analysis\Twitter_Data.csv")

print(df.shape)
df.head()

(162975, 1)


,clean_text
0,when modi promised “minimum government maximum...
1,talk all the nonsense and continue all the dra...
2,what did just say vote for modi welcome bjp t...
3,asking his supporters prefix chowkidar their n...
4,answer who among these the most powerful world...


In [2]:
df['clean_text'].isnull().sum()

0

In [5]:
df['clean_text'] = df['clean_text'].astype(str)

In [14]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text)
    
    # lowercase
    text = text.lower()
    
    # remove urls
    text = re.sub(r'http\S+', '', text)
    
    # remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # remove punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
df['processed_text'] = df['clean_text'].apply(clean_text)

In [16]:
df[['clean_text', 'processed_text']].head(5)

,clean_text,processed_text
0,when modi promised “minimum government maximum...,modi promised minimum government maximum gover...
1,talk all the nonsense and continue all the dra...,talk nonsense continue drama vote modi
2,what did just say vote for modi welcome bjp t...,say vote modi welcome bjp told rahul main camp...
3,asking his supporters prefix chowkidar their n...,asking supporters prefix chowkidar names modi ...
4,answer who among these the most powerful world...,answer among powerful world leader today trump...


In [20]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = analyzer.polarity_scores(text)['compound']
    
    if score > 0.05:
        return "positive"
    elif score < -0.05:
        return "negative"
    else:
        return "neutral"

df['sentiment'] = df['processed_text'].apply(get_sentiment)

In [21]:
df[['processed_text', 'sentiment']].head()

,processed_text,sentiment
0,modi promised minimum government maximum gover...,positive
1,talk nonsense continue drama vote modi,negative
2,say vote modi welcome bjp told rahul main camp...,positive
3,asking supporters prefix chowkidar names modi ...,positive
4,answer among powerful world leader today trump...,positive


In [22]:
df['sentiment'].value_counts()

sentiment
positive    77504
negative    50022
neutral     35449
Name: count, dtype: int64

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X = tfidf.fit_transform(df['processed_text'])

print(X.shape)

(162975, 5000)


In [24]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(
    n_components=4,        # number of topics
    random_state=42,
    learning_method='batch'
)

lda.fit(X)

LatentDirichletAllocation(n_components=4, random_state=42)

In [25]:
words = tfidf.get_feature_names_out()

for i, topic in enumerate(lda.components_):
    top_words = [words[j] for j in topic.argsort()[-10:]]
    print(f"Topic {i}: {top_words}")

Topic 0: ['power', 'nehru', 'satellite', 'mission', 'scientists', 'drdo', 'credit', 'space', 'india', 'modi']
Topic 1: ['nation', 'sir', 'chowkidar', 'prime', 'minister', 'hai', 'vote', 'india', 'narendra', 'modi']
Topic 2: ['india', 'people', 'rahul', 'modis', 'win', 'nirav', 'election', 'like', 'bjp', 'modi']
Topic 3: ['poor', 'money', 'years', 'like', 'dont', 'govt', 'congress', 'india', 'people', 'modi']


In [27]:
topic_values = lda.transform(X)

In [28]:
df['topic'] = topic_values.argmax(axis=1)

In [29]:
df[['processed_text', 'topic']].head()

,processed_text,topic
0,modi promised minimum government maximum gover...,3
1,talk nonsense continue drama vote modi,3
2,say vote modi welcome bjp told rahul main camp...,1
3,asking supporters prefix chowkidar names modi ...,2
4,answer among powerful world leader today trump...,0


In [30]:
topic_map = {
    0: "National Achievements / Defense",
    1: "Political Campaign / Leadership",
    2: "Elections & Party Politics",
    3: "Economic & Public Issues"
}

df['topic_name'] = df['topic'].map(topic_map)

In [ ]:
df['topic_name'].value_counts() #what are most tweets about

topic_name
Economic & Public Issues           58274
Elections & Party Politics         45785
National Achievements / Defense    30599
Political Campaign / Leadership    28317
Name: count, dtype: int64

In [32]:
df['sentiment'].value_counts(normalize=True) * 100

sentiment
positive    47.555760
negative    30.693051
neutral     21.751189
Name: proportion, dtype: float64

In [33]:
import pandas as pd

pd.crosstab(
    df['topic_name'], 
    df['sentiment'], 
    normalize='index'
) * 100

sentiment,negative,neutral,positive
topic_name,,,
Economic & Public Issues,37.891684,16.774205,45.334111
Elections & Party Politics,33.897565,21.393469,44.708966
National Achievements / Defense,28.853884,22.618386,48.527730
Political Campaign / Leadership,12.684960,31.634707,55.680333


In [34]:
df.to_csv("processed_data.csv", index=False)